In [17]:
# --- Setup: imports and loading the data ---
import pandas as pd
import numpy as np

# Load the raw pitch-level data (same file the daily cron job keeps updated)
df = pd.read_parquet("statcast_2026.parquet")

print(f"Loaded {len(df):,} pitches")
print(f"Date range: {df['game_date'].min()} to {df['game_date'].max()}")

Loaded 538,039 pitches
Date range: 2026-03-25 00:00:00 to 2026-08-13 00:00:00


In [18]:
# For each pitcher, count how many times they threw each individual
# pitch type this season. We use pitch_type (specific, like FF/SL/CH)
# here, not pitch_category (the broad fastball/breaking/offspeed bucket
# from Phase 2) -- for combo scoring we need the actual distinct pitches
# in a pitcher's toolkit, not the broad category they fall into.
pitch_counts = df.groupby(['pitcher', 'pitch_type']).size().reset_index(name='n_thrown')

MIN_PITCH_COUNT = 100
arsenal = pitch_counts[pitch_counts['n_thrown'] >= MIN_PITCH_COUNT].copy()

arsenal_size = arsenal.groupby('pitcher').size().rename('arsenal_size')
print(arsenal_size.describe())
print(f"\nTotal pitchers with a qualifying arsenal: {arsenal['pitcher'].nunique()}")
print(f"Pitchers with fewer than 3 qualifying pitches (can't form a 3-pitch combo): "
      f"{(arsenal_size < 3).sum()}")

count    530.000000
mean       3.133962
std        1.480490
min        1.000000
25%        2.000000
50%        3.000000
75%        4.000000
max        7.000000
Name: arsenal_size, dtype: float64

Total pitchers with a qualifying arsenal: 530
Pitchers with fewer than 3 qualifying pitches (can't form a 3-pitch combo): 206


In [19]:
from itertools import combinations

# For each pitcher, take their list of qualifying pitch types (100+ thrown)
# and generate every possible 3-pitch combination from it.
# Example: a pitcher with [FF, SL, CH, CU] gets 4 combos:
# (FF,SL,CH), (FF,SL,CU), (FF,CH,CU), (SL,CH,CU)

# First, build a dict of pitcher -> list of their qualifying pitch types
pitcher_arsenals = arsenal.groupby('pitcher')['pitch_type'].apply(list).to_dict()

# Only pitchers with 3+ qualifying pitches can form a combo at all
combo_rows = []
for pitcher_id, pitches in pitcher_arsenals.items():
    if len(pitches) < 3:
        continue  # can't form a 3-pitch combo, skip (the 203 we already identified)
    # itertools.combinations gives every unique 3-pitch subset, order doesn't matter
    for combo in combinations(sorted(pitches), 3):
        combo_rows.append({
            'pitcher': pitcher_id,
            'pitch_1': combo[0],
            'pitch_2': combo[1],
            'pitch_3': combo[2],
        })

all_combos = pd.DataFrame(combo_rows)

print(f"Total pitcher-combo rows to score: {len(all_combos):,}")
print(f"Pitchers represented: {all_combos['pitcher'].nunique()}")
all_combos.head(10)

Total pitcher-combo rows to score: 2,037
Pitchers represented: 324


,pitcher,pitch_1,pitch_2,pitch_3
0,453286,CH,CU,FF
1,453286,CH,CU,SL
2,453286,CH,FF,SL
3,453286,CU,FF,SL
4,500779,CH,CU,FF
5,500779,CH,CU,SI
6,500779,CH,FF,SI
7,500779,CU,FF,SI
8,518585,FF,FS,SL
9,518876,CH,CU,FC


In [20]:
# Sanity check delta_run_exp's sign convention and overall behavior
# before using it as our core success metric
print(df['delta_run_exp'].describe())

# Check it against known good/bad outcomes to confirm the direction:
# a strikeout should show a clearly NEGATIVE average (helped the pitcher),
# a home run should show a clearly POSITIVE average (helped the batter)
print("\nAvg delta_run_exp on strikeout pitches:")
print(df[df['events'] == 'strikeout']['delta_run_exp'].mean())

print("\nAvg delta_run_exp on home_run pitches:")
print(df[df['events'] == 'home_run']['delta_run_exp'].mean())

count    536346.0
mean    -0.000003
std      0.224515
min        -0.636
25%        -0.064
50%        -0.035
75%         0.043
max         2.699
Name: delta_run_exp, dtype: Float64

Avg delta_run_exp on strikeout pitches:
-0.22113185224135665

Avg delta_run_exp on home_run pitches:
1.5162185792349727


In [21]:
# Recreate the swing/whiff tagging from Phase 2, since this is a fresh
# notebook that doesn't have it yet.
swing_descriptions = [
    'foul', 'hit_into_play', 'swinging_strike', 'swinging_strike_blocked',
    'foul_tip', 'foul_bunt', 'missed_bunt'
]
whiff_descriptions = ['swinging_strike', 'swinging_strike_blocked', 'foul_tip']

df['is_swing'] = df['description'].isin(swing_descriptions)
df['is_whiff'] = df['description'].isin(whiff_descriptions)

In [22]:
# For every (pitcher, combo, handedness) row, we need the average
# delta_run_exp across all pitches that pitcher threw using one of
# those 3 pitch types, against a batter of that handedness.

# Step 1: pre-compute average delta_run_exp per (pitcher, pitch_type, stand)
# -- this is the building block; we'll look this up 3 times per combo
# (once per pitch in the combo) rather than recalculating from scratch
# for every single one of the 2,018 combo rows.
pitch_performance = (
    df.groupby(['pitcher', 'pitch_type', 'stand'])
    .agg(
        avg_run_value=('delta_run_exp', 'mean'),
        n_pitches=('delta_run_exp', 'count'),
        whiff_rate=('is_whiff', 'mean'),  # reuse the columns we tagged back in Phase 2 setup
    )
    .reset_index()
)

pitch_performance.head()

,pitcher,pitch_type,stand,avg_run_value,n_pitches,whiff_rate
0,434378,CH,L,-0.059667,6,0.000000
1,434378,CH,R,0.0115,2,0.000000
2,434378,CU,L,0.047455,11,0.090909
3,434378,CU,R,0.136,3,0.000000
4,434378,FF,L,-0.01605,20,0.000000


In [23]:
# How bad is the sample-size problem across the board?
print(pitch_performance['n_pitches'].describe())

# Specifically: how many (pitcher, pitch_type, stand) rows have a
# dangerously small sample -- say, under 20 pitches?
print(f"\nRows with fewer than 20 pitches: {(pitch_performance['n_pitches'] < 20).sum()} "
      f"out of {len(pitch_performance)} total")
print(f"Rows with fewer than 10 pitches: {(pitch_performance['n_pitches'] < 10).sum()}")

count       6949.0
mean     77.134408
std      97.882802
min            1.0
25%           11.0
50%           39.0
75%          109.0
max          838.0
Name: n_pitches, dtype: Float64

Rows with fewer than 20 pitches: 2444 out of 6949 total
Rows with fewer than 10 pitches: 1581


In [24]:
# Step 1: pitcher's overall average for each pitch type, regardless of
# handedness -- this becomes our shrinkage target (more specific/relevant
# than league average, since it's still about THIS pitcher's THIS pitch)
pitcher_pitch_overall = (
    df.groupby(['pitcher', 'pitch_type'])
    .agg(overall_avg_run_value=('delta_run_exp', 'mean'),
         overall_n=('delta_run_exp', 'count'))
    .reset_index()
)

# Step 2: merge that onto our handedness-split table
pitch_performance = pitch_performance.merge(
    pitcher_pitch_overall, on=['pitcher', 'pitch_type'], how='left'
)

# Step 3: apply shrinkage -- same formula concept as Phase 2's xwOBA fix.
# The handedness-specific average gets weighted by its own sample size (n);
# the pitcher's overall (handedness-agnostic) average acts as the "prior,"
# weighted by K. Small-sample handedness splits get pulled hard toward
# the pitcher's overall number; well-sampled ones barely move.
K = 30  # treat the pitcher's overall average as equivalent to 30 pitches of evidence

pitch_performance['avg_run_value_shrunk'] = (
    (pitch_performance['n_pitches'] * pitch_performance['avg_run_value'] +
     K * pitch_performance['overall_avg_run_value'])
    / (pitch_performance['n_pitches'] + K)
)

pitch_performance[['pitcher', 'pitch_type', 'stand', 'n_pitches',
                    'avg_run_value', 'overall_avg_run_value', 'avg_run_value_shrunk']].head(10)

,pitcher,pitch_type,stand,n_pitches,avg_run_value,overall_avg_run_value,avg_run_value_shrunk
0,434378,CH,L,6,-0.059667,-0.041875,-0.04484
1,434378,CH,R,2,0.0115,-0.041875,-0.038539
2,434378,CU,L,11,0.047455,0.066429,0.061338
3,434378,CU,R,3,0.136,0.066429,0.072753
4,434378,FF,L,20,-0.01605,-0.005059,-0.009455
5,434378,FF,R,14,0.010643,-0.005059,-0.000063
6,434378,SL,L,15,0.137733,0.077045,0.097275
7,434378,SL,R,7,-0.053,0.077045,0.052442
8,434378,ST,L,1,0.292,0.064,0.071355
9,434378,ST,R,1,-0.164,0.064,0.056645


In [25]:
# For each combo (3 pitch types), look up the shrunk run value for each
# of the 3 pitches, separately for L and R batters, then average the 3
# together to get one combo-level score per handedness.

def score_combo(row, stand):
    pitcher_id = row['pitcher']
    pitches = [row['pitch_1'], row['pitch_2'], row['pitch_3']]

    # Look up each pitch's shrunk run value for this pitcher, against
    # this specific handedness
    values = []
    for pt in pitches:
        match = pitch_performance[
            (pitch_performance['pitcher'] == pitcher_id) &
            (pitch_performance['pitch_type'] == pt) &
            (pitch_performance['stand'] == stand)
        ]
        if not match.empty:
            values.append(match['avg_run_value_shrunk'].values[0])

    # Combo score = simple average of the 3 pitches' shrunk run values
    # (a straightforward starting approach -- we can weight this by
    # usage rate later if we want a pitcher's most-thrown pitch to count
    # more heavily)
    return np.mean(values) if values else np.nan

# Score every combo against lefties and righties separately
all_combos['combo_score_vs_L'] = all_combos.apply(lambda r: score_combo(r, 'L'), axis=1)
all_combos['combo_score_vs_R'] = all_combos.apply(lambda r: score_combo(r, 'R'), axis=1)

all_combos.head(10)

,pitcher,pitch_1,pitch_2,pitch_3,combo_score_vs_L,combo_score_vs_R
0,453286,CH,CU,FF,0.006379,0.000815
1,453286,CH,CU,SL,0.018685,0.019008
2,453286,CH,FF,SL,0.017042,0.000780
3,453286,CU,FF,SL,0.023684,0.011766
4,500779,CH,CU,FF,0.008907,0.002401
5,500779,CH,CU,SI,0.010592,0.009507
6,500779,CH,FF,SI,0.002278,0.014127
7,500779,CU,FF,SI,-0.010390,0.010367
8,518585,FF,FS,SL,-0.011408,-0.002091
9,518876,CH,CU,FC,0.010223,0.004373


In [26]:
# For each pitcher, find their BEST 3-pitch combo (lowest/most negative
# score = best for the pitcher) separately against lefties and righties

best_vs_L = (
    all_combos.sort_values('combo_score_vs_L')
    .groupby('pitcher')
    .first()  # lowest combo_score_vs_L per pitcher, since we sorted ascending
    .reset_index()[['pitcher', 'pitch_1', 'pitch_2', 'pitch_3', 'combo_score_vs_L']]
)

best_vs_R = (
    all_combos.sort_values('combo_score_vs_R')
    .groupby('pitcher')
    .first()
    .reset_index()[['pitcher', 'pitch_1', 'pitch_2', 'pitch_3', 'combo_score_vs_R']]
)

print("Best combo per pitcher vs LEFTIES:")
print(best_vs_L.head(10))

print("\nBest combo per pitcher vs RIGHTIES:")
print(best_vs_R.head(10))

Best combo per pitcher vs LEFTIES:
   pitcher pitch_1 pitch_2 pitch_3  combo_score_vs_L
0   453286      CH      CU      FF          0.006379
1   500779      CU      FF      SI         -0.010390
2   518585      FF      FS      SL         -0.011408
3   518876      CU      SI      SL         -0.002891
4   519242      CH      FF      SI         -0.016282
5   527048      CH      CU      SI         -0.005794
6   542888      FC      FF      ST         -0.006745
7   543037      CH      FF      SL         -0.011261
8   543135      CU      FC      FS         -0.006089
9   543243      FF      SI      ST         -0.022869

Best combo per pitcher vs RIGHTIES:
   pitcher pitch_1 pitch_2 pitch_3  combo_score_vs_R
0   453286      CH      FF      SL          0.000780
1   500779      CH      CU      FF          0.002401
2   518585      FF      FS      SL         -0.002091
3   518876      CH      CU      SL         -0.007124
4   519242      CH      SI      SL         -0.017943
5   527048      CH      CU 

In [27]:
# Step 1: total pitches each pitcher threw to each handedness (across
# their FULL arsenal, not just the 100+ qualifying ones) -- this is the
# denominator for calculating what % each pitch represents of their mix
total_by_hand = (
    pitch_performance.groupby(['pitcher', 'stand'])['n_pitches']
    .sum()
    .rename('total_pitches_to_hand')
    .reset_index()
)

pitch_performance = pitch_performance.merge(total_by_hand, on=['pitcher', 'stand'], how='left')

# Step 2: usage rate = what % of pitches to this handedness was this
# specific pitch type
pitch_performance['usage_rate'] = (
    pitch_performance['n_pitches'] / pitch_performance['total_pitches_to_hand']
)

pitch_performance[['pitcher', 'pitch_type', 'stand', 'n_pitches', 'usage_rate']].head(8)

,pitcher,pitch_type,stand,n_pitches,usage_rate
0,434378,CH,L,6,0.113208
1,434378,CH,R,2,0.074074
2,434378,CU,L,11,0.207547
3,434378,CU,R,3,0.111111
4,434378,FF,L,20,0.377358
5,434378,FF,R,14,0.518519
6,434378,SL,L,15,0.283019
7,434378,SL,R,7,0.259259


In [28]:
def score_combo_weighted(row, stand):
    pitcher_id = row['pitcher']
    pitches = [row['pitch_1'], row['pitch_2'], row['pitch_3']]

    weighted_sum = 0
    weight_total = 0
    for pt in pitches:
        match = pitch_performance[
            (pitch_performance['pitcher'] == pitcher_id) &
            (pitch_performance['pitch_type'] == pt) &
            (pitch_performance['stand'] == stand)
        ]
        if not match.empty:
            value = match['avg_run_value_shrunk'].values[0]
            # Weight by n_pitches directly -- mathematically identical to
            # weighting by usage_rate here, since usage_rate's denominator
            # (total_pitches_to_hand) is the same constant across all 3
            # pitches in a combo for a given pitcher/handedness, so it
            # cancels out. n_pitches is simpler to use directly.
            weight = match['n_pitches'].values[0]
            weighted_sum += value * weight
            weight_total += weight

    return weighted_sum / weight_total if weight_total > 0 else np.nan

all_combos['combo_score_vs_L_weighted'] = all_combos.apply(lambda r: score_combo_weighted(r, 'L'), axis=1)
all_combos['combo_score_vs_R_weighted'] = all_combos.apply(lambda r: score_combo_weighted(r, 'R'), axis=1)

all_combos.head(10)

,pitcher,pitch_1,pitch_2,pitch_3,combo_score_vs_L,combo_score_vs_R,combo_score_vs_L_weighted,combo_score_vs_R_weighted
0,453286,CH,CU,FF,0.006379,0.000815,0.007149,-0.013674
1,453286,CH,CU,SL,0.018685,0.019008,0.016626,0.024836
2,453286,CH,FF,SL,0.017042,0.000780,0.013088,-0.000338
3,453286,CU,FF,SL,0.023684,0.011766,0.018017,0.002977
4,500779,CH,CU,FF,0.008907,0.002401,-0.003621,0.003947
5,500779,CH,CU,SI,0.010592,0.009507,-0.004088,0.008339
6,500779,CH,FF,SI,0.002278,0.014127,-0.011943,0.011331
7,500779,CU,FF,SI,-0.010390,0.010367,-0.012918,0.009345
8,518585,FF,FS,SL,-0.011408,-0.002091,-0.008035,-0.001857
9,518876,CH,CU,FC,0.010223,0.004373,0.009220,0.004550


In [29]:
# Find each pitcher's best combo under BOTH weighted and unweighted
# scoring, then check how often the two methods actually disagree

best_vs_L_weighted = (
    all_combos.sort_values('combo_score_vs_L_weighted')
    .groupby('pitcher').first().reset_index()
    [['pitcher', 'pitch_1', 'pitch_2', 'pitch_3', 'combo_score_vs_L_weighted']]
    .rename(columns={'pitch_1': 'w_pitch_1', 'pitch_2': 'w_pitch_2', 'pitch_3': 'w_pitch_3'})
)

# Merge against the unweighted winners from before, comparing pitch-by-pitch
comparison = best_vs_L.merge(best_vs_L_weighted, on='pitcher')

# A combo "changed" if the set of 3 pitches differs, regardless of order
def combo_changed(row):
    unweighted_set = {row['pitch_1'], row['pitch_2'], row['pitch_3']}
    weighted_set = {row['w_pitch_1'], row['w_pitch_2'], row['w_pitch_3']}
    return unweighted_set != weighted_set

comparison['changed'] = comparison.apply(combo_changed, axis=1)

print(f"Pitchers whose BEST combo vs lefties changed after weighting: "
      f"{comparison['changed'].sum()} out of {len(comparison)} "
      f"({comparison['changed'].mean():.1%})")

comparison[comparison['changed']].head(10)

Pitchers whose BEST combo vs lefties changed after weighting: 73 out of 324 (22.5%)


,pitcher,pitch_1,pitch_2,pitch_3,combo_score_vs_L,w_pitch_1,w_pitch_2,w_pitch_3,combo_score_vs_L_weighted,changed
5,527048,CH,CU,SI,-0.005794,CH,CU,FC,-0.008187,True
7,543037,CH,FF,SL,-0.011261,FF,KC,SL,-0.013991,True
8,543135,CU,FC,FS,-0.006089,FC,FS,SI,-0.012228,True
11,547179,CH,CU,SL,0.010692,CU,SL,ST,0.009887,True
14,554430,FC,FS,SI,-0.011769,FS,SI,ST,-0.010446,True
15,571510,CU,FF,SL,-0.009049,CH,CU,SL,-0.014554,True
24,592662,FF,KC,SL,-0.007620,CH,KC,SL,-0.009555,True
26,592791,CH,CU,FF,0.007139,CU,FF,ST,0.005386,True
35,605400,CH,FF,KC,0.006287,CH,FC,KC,-0.001304,True
36,605488,FF,SL,ST,0.007217,CH,FF,ST,0.003805,True


In [30]:
from pybaseball import playerid_lookup

# Look up Yoshinobu Yamamoto's MLBAM ID by name
lookup_result = playerid_lookup('yamamoto', 'yoshinobu')
print(lookup_result)

  name_last name_first  key_mlbam key_retro  key_bbref  key_fangraphs  \
0  yamamoto  yoshinobu     808967  yamay001  yamamyo01          33825   

   mlb_played_first  mlb_played_last  
0            2024.0           2026.0  


In [31]:
# Check Yamamoto's actual pitch mix and whether he qualifies for combo scoring
yamamoto_id = 808967

print("Yamamoto's qualifying arsenal:")
print(arsenal[arsenal['pitcher'] == yamamoto_id])

print("\nHis combo options and scores vs lefties and righties:")
yamamoto_combos = all_combos[all_combos['pitcher'] == yamamoto_id].sort_values('combo_score_vs_L_weighted')
print(yamamoto_combos[['pitch_1', 'pitch_2', 'pitch_3', 'combo_score_vs_L_weighted', 'combo_score_vs_R_weighted']])

Yamamoto's qualifying arsenal:
      pitcher pitch_type  n_thrown
3671   808967         CU       256
3672   808967         FC       307
3673   808967         FF       548
3674   808967         FS       560
3675   808967         SI       261
3676   808967         SL       131

His combo options and scores vs lefties and righties:
     pitch_1 pitch_2 pitch_3  combo_score_vs_L_weighted  \
2014      FF      SI      SL                  -0.024999   
2012      FF      FS      SI                  -0.022794   
2015      FS      SI      SL                  -0.022701   
2001      CU      FF      SI                  -0.022029   
2013      FF      FS      SL                  -0.021775   
2005      CU      SI      SL                  -0.020732   
2002      CU      FF      SL                  -0.020567   
2007      FC      FF      SI                  -0.020447   
2000      CU      FF      FS                  -0.020349   
2003      CU      FS      SI                  -0.020305   
2006      FC      FF